In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()

engine = create_engine(
    f"postgresql://{os.getenv('DB_USERNAME')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

plt.style.use('seaborn-v0_8')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (10, 6)

print("Ready")

Ready


In [3]:
# Loading datasets
reg = pd.read_csv('../datasets/beneficiaries_registration003.csv')
dist = pd.read_csv('../datasets/distribution_records004.csv')

print("Registration shape:", reg.shape)
print("Distribution shape:", dist.shape)

print("\nRegistration columns:", reg.columns.tolist())
print("Distribution columns:", dist.columns.tolist())

Registration shape: (20, 9)
Distribution shape: (20, 8)

Registration columns: ['beneficiary_id', 'full_name', 'age', 'gender', 'province', 'district', 'household_size', 'registration_date', 'phone']
Distribution columns: ['distribution_id', 'beneficiary_id', 'distribution_date', 'food_package_kg', 'enumerator', 'status', 'monthly_income_usd', 'vulnerability_score']


In [4]:
# Check if beneficiary_id is unique in registration, nunique - count of unique values
print("Unique IDs in registration:", reg['beneficiary_id'].nunique())
print("Total rows in registration:", len(reg)) # if both equal every ID is unique

# Check if beneficiary_id is unique in distribution
print("\nUnique IDs in distribution:", dist['beneficiary_id'].nunique())
print("Total rows in distribution:", len(dist))

# Check which IDs are in distribution but not registration, set - to see the difference
dist_only = set(dist['beneficiary_id']) - set(reg['beneficiary_id'])
reg_only = set(reg['beneficiary_id']) - set(dist['beneficiary_id'])

print("\nIn distribution but not registration:", dist_only)
print("In registration but not distribution:", reg_only)

Unique IDs in registration: 20
Total rows in registration: 20

Unique IDs in distribution: 20
Total rows in distribution: 20

In distribution but not registration: {24, 21, 22, 23}
In registration but not distribution: {17, 11, 20, 6}


In [5]:
# Inner join —> only beneficiaries who are both registered AND received food
# only matching IDs (1-5, 7-10, 12-16, 18-19) = 16 rows

inner_merge = pd.merge(
    reg,
    dist,
    on='beneficiary_id',
    how='inner'
)

print("Inner join shape:", inner_merge.shape)
print("\nColumns:", inner_merge.columns.tolist())
print(inner_merge[['beneficiary_id', 'full_name', 'province', 'food_package_kg', 'status']])

Inner join shape: (16, 16)

Columns: ['beneficiary_id', 'full_name', 'age', 'gender', 'province', 'district', 'household_size', 'registration_date', 'phone', 'distribution_id', 'distribution_date', 'food_package_kg', 'enumerator', 'status', 'monthly_income_usd', 'vulnerability_score']
    beneficiary_id        full_name  province  food_package_kg     status
0                1     Ahmad Karimi     Kabul               25  Completed
1                2     Fatima Noori     Kabul               25  Completed
2                3   Mohammad Yusuf  Kandahar               25  Completed
3                4   Zainab Hussain     Herat               25  Completed
4                5      Abdul Karim     Kabul               25  Completed
5                7    Khalid Ahmadi  Kandahar               25  Completed
6                8     Sadia Rahimi     Kabul               25  Completed
7                9    Najiba Karimi     Balkh               25  Completed
8               10     Omar Sharifi     Kabul   

In [6]:
# loads tables to Postgres 
reg.to_sql('beneficiary_registration', engine, if_exists='replace', index=False)
dist.to_sql('distribution_records', engine, if_exists='replace', index=False)
print("Both tables loaded successfully")

Both tables loaded successfully


In [7]:
# SQL
# INNER JOIN ... ON —> connects two tables where the key matches. Only returns rows where match exists in both tables.
query = """
SELECT 
    r.beneficiary_id,
    r.full_name,
    r.province,
    r.household_size,
    d.food_package_kg,
    d.distribution_date,
    d.status,
    d.monthly_income_usd
FROM beneficiary_registration r
INNER JOIN distribution_records d
    ON r.beneficiary_id = d.beneficiary_id
ORDER BY r.beneficiary_id
"""
result = pd.read_sql(query, engine)
print(result)

    beneficiary_id        full_name  province  household_size  \
0                1     Ahmad Karimi     Kabul               6   
1                2     Fatima Noori     Kabul               4   
2                3   Mohammad Yusuf  Kandahar               3   
3                4   Zainab Hussain     Herat               5   
4                5      Abdul Karim     Kabul               7   
5                7    Khalid Ahmadi  Kandahar               9   
6                8     Sadia Rahimi     Kabul               2   
7                9    Najiba Karimi     Balkh               4   
8               10     Omar Sharifi     Kabul               6   
9               12  Habibullah Khan  Kandahar               8   
10              13    Razia Sultani     Herat               5   
11              14   Bismillah Omar     Kabul               7   
12              15     Laila Ahmadi     Balkh               2   
13              16     Qasim Wardak     Kabul               5   
14              18    Jaw

In [8]:
# Left join —> all registered beneficiaries, with distribution info where available
# all registration + matches from distribution = 20 rows
left_merge = pd.merge(
    reg,
    dist,
    on='beneficiary_id',
    how='left'
)

print("Left join shape:", left_merge.shape)

# Find beneficiaries who never received food
not_distributed = left_merge[left_merge['food_package_kg'].isnull()]
print("\nBeneficiaries who never received food:")
print(not_distributed[['beneficiary_id', 'full_name', 'province', 'household_size']])

Left join shape: (20, 16)

Beneficiaries who never received food:
    beneficiary_id       full_name province  household_size
5                6  Mariam Sultani    Herat               3
10              11  Freshta Ahmadi    Balkh               3
16              17    Maryam Noori    Herat               4
19              20   Shirin Ahmadi    Kabul               4


In [9]:
# SQL
query = """
SELECT 
    r.beneficiary_id,
    r.full_name,
    r.province,
    r.household_size,
    d.food_package_kg,
    d.distribution_date,
    d.status
FROM beneficiary_registration r
LEFT JOIN distribution_records d
    ON r.beneficiary_id = d.beneficiary_id
ORDER BY r.beneficiary_id
"""
result = pd.read_sql(query, engine)
print("Left join shape:", result.shape)

# Find missing distributions in SQL
query_missing = """
SELECT 
    r.beneficiary_id,
    r.full_name,
    r.province,
    r.household_size
FROM beneficiary_registration r
LEFT JOIN distribution_records d
    ON r.beneficiary_id = d.beneficiary_id
WHERE d.beneficiary_id IS NULL
ORDER BY r.beneficiary_id
"""
missing = pd.read_sql(query_missing, engine)
print("\nBeneficiaries who never received food:")
print(missing)

Left join shape: (20, 7)

Beneficiaries who never received food:
   beneficiary_id       full_name province  household_size
0               6  Mariam Sultani    Herat               3
1              11  Freshta Ahmadi    Balkh               3
2              17    Maryam Noori    Herat               4
3              20   Shirin Ahmadi    Kabul               4


In [10]:
# Right join —> all distribution records, with registration info where available
# Right join —> all distribution + matches from registration = 20 rows
right_merge = pd.merge(
    reg,
    dist,
    on='beneficiary_id',
    how='right'
)

print("Right join shape:", right_merge.shape)

# Find distributions without registration
not_registered = right_merge[right_merge['full_name'].isnull()]
print("\nBeneficiaries who received food but are NOT registered:")
print(not_registered[['beneficiary_id', 'food_package_kg', 'distribution_date', 'status']])

Right join shape: (20, 16)

Beneficiaries who received food but are NOT registered:
    beneficiary_id  food_package_kg distribution_date     status
16              21               25        2024-02-11  Completed
17              22               25        2024-02-11  Completed
18              23               25        2024-02-12  Completed
19              24               25        2024-02-12  Completed


In [11]:
# SQL
query = """
SELECT
    d.beneficiary_id,
    r.full_name,
    d.food_package_kg,
    d.distribution_date,
    d.status
FROM beneficiary_registration r
RIGHT JOIN distribution_records d
    ON r.beneficiary_id = d.beneficiary_id
ORDER BY d.beneficiary_id
"""
result = pd.read_sql(query, engine)
print("Right join shape:", result.shape)

# Find unregistered distributions
query_unregistered = """
SELECT
    d.beneficiary_id,
    d.food_package_kg,
    d.distribution_date,
    d.status
FROM beneficiary_registration r
RIGHT JOIN distribution_records d
    ON r.beneficiary_id = d.beneficiary_id
WHERE r.beneficiary_id IS NULL
ORDER BY d.beneficiary_id
"""
unregistered = pd.read_sql(query_unregistered, engine)
print("\nUnregistered beneficiaries who received food:")
print(unregistered)

Right join shape: (20, 5)

Unregistered beneficiaries who received food:
   beneficiary_id  food_package_kg distribution_date     status
0              21               25        2024-02-11  Completed
1              22               25        2024-02-11  Completed
2              23               25        2024-02-12  Completed
3              24               25        2024-02-12  Completed


In [12]:
# Full outer join —> everything from both tables = 24 rows
full_merge = pd.merge(
    reg,
    dist,
    on='beneficiary_id',
    how='outer'
)

print("Full outer join shape:", full_merge.shape)

# Summary of match status
full_merge['match_status'] = 'Matched'
full_merge.loc[full_merge['full_name'].isnull(), 'match_status'] = 'Distribution only - not registered'
full_merge.loc[full_merge['food_package_kg'].isnull(), 'match_status'] = 'Registered only - no distribution'

print("\nMatch status summary:")
print(full_merge['match_status'].value_counts())

print("\nFull breakdown:")
print(full_merge[['beneficiary_id', 'full_name', 'food_package_kg', 'match_status']])

Full outer join shape: (24, 16)

Match status summary:
match_status
Matched                               16
Registered only - no distribution      4
Distribution only - not registered     4
Name: count, dtype: int64

Full breakdown:
    beneficiary_id        full_name  food_package_kg  \
0                1     Ahmad Karimi             25.0   
1                2     Fatima Noori             25.0   
2                3   Mohammad Yusuf             25.0   
3                4   Zainab Hussain             25.0   
4                5      Abdul Karim             25.0   
5                6   Mariam Sultani              NaN   
6                7    Khalid Ahmadi             25.0   
7                8     Sadia Rahimi             25.0   
8                9    Najiba Karimi             25.0   
9               10     Omar Sharifi             25.0   
10              11   Freshta Ahmadi              NaN   
11              12  Habibullah Khan             25.0   
12              13    Razia Sultani   

In [ ]:
# SQL
query = """
SELECT
    COALESCE(r.beneficiary_id, d.beneficiary_id) as beneficiary_id,
    r.full_name,
    r.province,
    d.food_package_kg,
    d.status,
    CASE
        WHEN r.beneficiary_id IS NULL THEN 'Distribution only - not registered'
        WHEN d.beneficiary_id IS NULL THEN 'Registered only - no distribution'
        ELSE 'Matched'
    END as match_status
FROM beneficiary_registration r
FULL OUTER JOIN distribution_records d
    ON r.beneficiary_id = d.beneficiary_id
ORDER BY beneficiary_id
"""
result = pd.read_sql(query, engine)
print("Full outer join shape:", result.shape)
print("\nMatch status summary:")
print(result['match_status'].value_counts())

Full outer join shape: (24, 6)

Match status summary:
match_status
Matched                               16
Registered only - no distribution      4
Distribution only - not registered     4
Name: count, dtype: int64


Immediate action:

Field team must locate all 4 individuals and complete their registration forms.  
Verify their eligibility against program targeting criteria, do they actually qualify for assistance?  
If eligible, register and document properly  
If ineligible, flag for recovery or write off with justification  

Process change:

Implement a no registration, no distribution policy, enumerators cannot distribute food without a valid beneficiary ID in the system  
Monitor → Find issue → Evaluate cause → Act → Learn → Improve
